In [ ]:
from serpapi import GoogleSearch

API_KEY = "YOUR_SERPAPI_KEY"

params = {
    "engine": "google",
    "q": 'site:linkedin.com/in "HR" "Jio"',
    "api_key": API_KEY,
    "num": 10
}

search = GoogleSearch(params)
results = search.get_dict()

for r in results.get("organic_results", []):
    print(r.get("link"))

https://in.linkedin.com/in/harjeetkhanduja
https://in.linkedin.com/in/mustafa-m-merchant
https://in.linkedin.com/in/manishsinghhr
https://in.linkedin.com/in/hr1505
https://in.linkedin.com/in/rahulmu
https://in.linkedin.com/in/sanjeev-varshney-0114b210
https://in.linkedin.com/in/pallavi1401
https://in.linkedin.com/in/parthasamai75
https://in.linkedin.com/in/preethamsingh


In [6]:
import re

urls = [
    "https://in.linkedin.com/in/harjeetkhanduja",
    "https://in.linkedin.com/in/mustafa-m-merchant",
    "https://in.linkedin.com/in/manishsinghhr",
    "https://in.linkedin.com/in/hr1505",
    "https://in.linkedin.com/in/rahulmu",
    "https://in.linkedin.com/in/sanjeev-varshney-0114b210",
    "https://in.linkedin.com/in/pallavi1401",
    "https://in.linkedin.com/in/parthasamai75",
    "https://in.linkedin.com/in/preethamsingh"
]

def extract_name(url):
    match = re.search(r'linkedin\.com/in/([a-z0-9\-]+)', url)
    if not match:
        return None

    slug = match.group(1)

    # Remove trailing alphanumeric IDs e.g. -0114b210
    slug = re.sub(r'-[0-9a-f]{6,}$', '', slug)
    # Remove trailing numbers e.g. 75, 1505
    slug = re.sub(r'\d+$', '', slug)

    parts = slug.replace("-", " ").strip().split()

    # Remove noise words
    noise = {"hr", "m", "mr", "ms", "dr", "in"}
    parts = [p for p in parts if p not in noise]

    # Slug with no dashes — try to split camelcase style
    # e.g. harjeetkhanduja → too merged, keep as is
    if len(parts) == 1 and len(parts[0]) > 4:
        # Can't reliably split — return as single capitalized
        return parts[0].capitalize()

    if len(parts) >= 2:
        return " ".join(p.capitalize() for p in parts[:2])

    return None  # Not enough info

for url in urls:
    name = extract_name(url)
    slug = url.split('/in/')[1]
    if name:
        print(f"  [✓] {name:<25} ← {slug}")
    else:
        print(f"  [✗] SKIP                    ← {slug} (unclear name)")

  [✓] Harjeetkhanduja           ← harjeetkhanduja
  [✓] Mustafa Merchant          ← mustafa-m-merchant
  [✓] Manishsinghhr             ← manishsinghhr
  [✗] SKIP                    ← hr1505 (unclear name)
  [✓] Rahulmu                   ← rahulmu
  [✓] Sanjeev Varshney          ← sanjeev-varshney-0114b210
  [✓] Pallavi                   ← pallavi1401
  [✓] Parthasamai               ← parthasamai75
  [✓] Preethamsingh             ← preethamsingh


In [7]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_name_from_page(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=5)
        soup = BeautifulSoup(resp.text, "html.parser")
        title = soup.find("title")
        if title:
            # "Priya Sharma - HR Manager | LinkedIn"
            name = title.text.split("-")[0].strip()
            if len(name.split()) >= 2:
                return name
    except Exception as e:
        print(f"  [!] Error: {e}")
    return None

# Test on merged-name URLs only
test_urls = [
    "https://in.linkedin.com/in/harjeetkhanduja",
    "https://in.linkedin.com/in/manishsinghhr",
    "https://in.linkedin.com/in/rahulmu",
    "https://in.linkedin.com/in/pallavi1401",
]

for url in test_urls:
    name = fetch_name_from_page(url)
    print(f"  {name} ← {url.split('/in/')[1]}")

  Harjeet Khanduja ← harjeetkhanduja
  None ← manishsinghhr
  Rahul Mukherjee ← rahulmu
  None ← pallavi1401


In [8]:
from serpapi import GoogleSearch
from dotenv import load_dotenv
import os

load_dotenv()

params = {
    "engine": "google",
    "q": 'site:linkedin.com/in "HR" "Jio"',
    "api_key": os.getenv("SERPAPI_KEY"),
    "num": 10
}

search = GoogleSearch(params)
results = search.get_dict()

for r in results.get("organic_results", []):
    print(f"Title : {r.get('title')}")
    print(f"Link  : {r.get('link')}")
    print()

Title : Harjeet Khanduja - Jio
Link  : https://in.linkedin.com/in/harjeetkhanduja

Title : Mustafa Merchant - Vice President - HR at Jio Studios ...
Link  : https://in.linkedin.com/in/mustafa-m-merchant

Title : Manish Singh - Group Chief Human Resources Officer
Link  : https://in.linkedin.com/in/manishsinghhr

Title : Geeta B. - HR | JIO
Link  : https://in.linkedin.com/in/hr1505

Title : Rahul Mukherjee - Jio
Link  : https://in.linkedin.com/in/rahulmu

Title : Sanjeev Varshney - HR at Reliance Jio Infocomm Limited
Link  : https://in.linkedin.com/in/sanjeev-varshney-0114b210

Title : Pallavi Gupta - Senior Human Resources Manager at ...
Link  : https://in.linkedin.com/in/pallavi1401

Title : Capt.Partha Samai - Jio
Link  : https://in.linkedin.com/in/parthasamai75

Title : Preetham Singh - Vice President, CHRO
Link  : https://in.linkedin.com/in/preethamsingh



In [1]:
def search_via_serper(query: str, start: int = 0) -> list | None:
    try:
        page = (start // 10) + 1
        payload = {
            "q": query,
            "num": 10,
            "page": page,
            "hl": "en"
        }
        headers = {
            "X-API-KEY": SERPER_API_KEY,
            "Content-Type": "application/json"
        }
        response = requests.post(
            "https://google.serper.dev/search",
            json=payload,
            headers=headers
        )
        results = response.json()

        # 🔍 DEBUG — remove after testing
        print(f"  [DEBUG] Serper raw keys: {list(results.keys())}")
        print(f"  [DEBUG] organic count: {len(results.get('organic', []))}")
        for r in results.get("organic", []):
            print(f"  [DEBUG] → {r.get('link','')}")

        if "error" in results:
            print(f"  [Serper] Error: {results['error']}")
            return None

        organic = results.get("organic", [])
        return [{"title": r.get("title", ""), "link": r.get("link", "")} for r in organic]

    except Exception as e:
        print(f"  [Serper] Exception: {e}")
        return None

In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

SERPER_API_KEY = os.getenv("SERPER_API_KEY")

print(f"Key loaded: {SERPER_API_KEY[:8]}...")

payload = {
    "q": 'site:linkedin.com/in ("HR") "Jio"',
    "num": 10,
    "page": 1,
    "hl": "en"
}

headers = {
    "X-API-KEY": SERPER_API_KEY,
    "Content-Type": "application/json"
}

response = requests.post(
    "https://google.serper.dev/search",
    json=payload,
    headers=headers
)

print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


Key loaded: 0d7884ad...
Status: 200
Response: {'searchParameters': {'q': 'site:linkedin.com/in ("HR") "Jio"', 'hl': 'en', 'type': 'search', 'num': 10, 'page': 1, 'engine': 'google'}, 'organic': [{'title': 'Harjeet Khanduja - Jio - LinkedIn', 'link': 'https://in.linkedin.com/in/harjeetkhanduja', 'snippet': 'Harjeet is an International Author, Speaker, Poet, Inventor, Influencer, Professor of… · Experience: Jio ... 25 Influential HR Leaders. HR Association of India.', 'position': 1}, {'title': 'Rahul Mukherjee - CHRO at Jio I Digital Transformation - LinkedIn', 'link': 'https://in.linkedin.com/in/rahulmu', 'snippet': "HR Leader accountable for all HR Services for Jio. In this role, I've built and run the operations of a Automated High Touch HR Platform partnering business and ...", 'position': 2}, {'title': 'Manish Singh - Group Chief Human Resources Officer | LinkedIn', 'link': 'https://in.linkedin.com/in/manishsinghhr', 'snippet': '... HR function imbedded and complementing organizatio

In [4]:
import dns.resolver
records = dns.resolver.resolve("ril.com", 'MX')
for r in records:
    print(r)

1 gwjgsmtp020.ril.com.
1 GWJGSMTP60.ril.com.
1 gwsmtp040.ril.com.
1 GWJGSMTP30.ril.com.
1 gwsmtp050.ril.com.
1 gwjgsmtp10.ril.com.
1 gwsmtp30.ril.com.
1 gwhydsmtp10.ril.com.
1 gwhydsmtp20.ril.com.
1 gwsmtp20.ril.com.


In [5]:
import dns.resolver

domain = "ril.com"
try:
    records = dns.resolver.resolve(domain, 'MX')
    print(f"✅ Valid — {len(records)} MX records found")
except dns.resolver.NXDOMAIN:
    print("❌ NXDOMAIN — domain doesn't exist")
except dns.resolver.NoAnswer:
    print("❌ NoAnswer — no MX records")
except dns.resolver.Timeout:
    print("❌ Timeout")
except Exception as e:
    print(f"❌ Other error: {type(e).__name__}: {e}")

✅ Valid — 10 MX records found
